# 07 · Context parallelism: ring attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/distributed/07_ring_attention.ipynb) &nbsp; 📖 Concept deck: [Context parallelism](https://nilesh-salpe.github.io/visual-ai/distributed/06-context-parallelism/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** causal attention over a sequence split across N simulated GPUs: each keeps its query block and K/V blocks travel around a ring, combined with an online softmax. Exact, with memory per GPU ∝ sequence / N.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def full_causal_attention(Q, K, V):
    S = Q @ K.T / np.sqrt(Q.shape[1])
    S = np.where(np.tril(np.ones(S.shape, bool)), S, -np.inf)
    return softmax(S) @ V

## Task 1: Ring attention with an online softmax

Split Q, K, V into N equal blocks along the sequence; GPU i owns Q_i and starts with (K_i, V_i).
For step t = 0 … N−1, GPU i holds the K/V block j = (i − t) mod N (blocks move one GPU around the ring per step). It updates a running max m, sum ℓ and unnormalized output Õ for its queries:
* skip if j > i (entirely in the future, causal);
* on j = i, apply the causal mask within the block;
* otherwise attend to the whole block.

At the end O_i = Õ / ℓ. Return the full output (concatenate the N blocks) and the number of K/V blocks each GPU received.

In [ ]:
def ring_attention(Q, K, V, N):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_ring_attention(f):
    r = np.random.default_rng(1)
    for T, N in [(16, 4), (24, 3), (8, 8)]:
        Q, K, V = r.normal(size=(3, T, 8))
        O, recv = f(Q, K, V, N)
        assert np.allclose(O, full_causal_attention(Q, K, V), atol=1e-10), f"T={T}, N={N}: must equal full causal attention"
        assert recv == N - 1, f"each GPU receives N-1 K/V blocks, got {recv}"

check("ring_attention", _test_ring_attention, ring_attention)

## Task 2: Ring vs all-gather communication

Per GPU, per layer, for sequence length S, KV width w (= 2 · n_kv_heads · head_dim) and b bytes per value:
* **Ring** (Liu et al.): receives N − 1 K/V blocks of S/N tokens: (N−1)/N · S · w · b, overlapped with compute.
* **All-gather** (the variant Llama 3 uses): also receives (N−1)/N · S · w · b, but all at once before attention; simpler with arbitrary masks such as document masks, at the cost of exposed latency.

Return the bytes per GPU (same for both). The point of the exercise: the difference is *when*, not *how much*.

In [ ]:
def cp_comm_bytes(S, N, w, b=2):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_cp_comm_bytes(f):
    assert f(131072, 16, 2 * 8 * 128) == 15 / 16 * 131072 * 2048 * 2

check("cp_comm_bytes", _test_cp_comm_bytes, cp_comm_bytes)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Ring attention with an online softmax
def ring_attention_solution(Q, K, V, N):
    Qb, Kb, Vb = np.split(Q, N), np.split(K, N), np.split(V, N)
    bs, d = Qb[0].shape
    out = []
    for i in range(N):
        m = np.full(bs, -np.inf); l = np.zeros(bs); acc = np.zeros((bs, V.shape[1]))
        for t in range(N):
            j = (i - t) % N
            if j > i:
                continue
            S = Qb[i] @ Kb[j].T / np.sqrt(d)
            if j == i:
                S = np.where(np.tril(np.ones((bs, bs), bool)), S, -np.inf)
            m_new = np.maximum(m, S.max(1))
            P = np.exp(S - m_new[:, None]); corr = np.exp(m - m_new)
            l = l * corr + P.sum(1); acc = acc * corr[:, None] + P @ Vb[j]; m = m_new
        out.append(acc / l[:, None])
    return np.concatenate(out), N - 1

check("ring_attention (reference)", _test_ring_attention, ring_attention_solution)

In [ ]:
# @title Solution for Task 2: Ring vs all-gather communication
def cp_comm_bytes_solution(S, N, w, b=2):
    return (N - 1) / N * S * w * b

check("cp_comm_bytes (reference)", _test_cp_comm_bytes, cp_comm_bytes_solution)

---
## Experiments
**Load balance.** With a causal mask, GPU 0 (earliest tokens) skips most K/V blocks and GPU N−1 does the most work. Count attended blocks per GPU; real systems reorder tokens (e.g. give each GPU one early and one late chunk) to balance it.

In [ ]:
N = 8
work = [sum(1 for t in range(N) if (i - t) % N <= i) for i in range(N)]
print("K/V blocks attended per GPU with naive contiguous chunks:", work)
print("Llama 3 405B at 128K context: CP = 16 →", f"{cp_comm_bytes_solution(131072, 16, 2 * 8 * 128) / 2**20:.0f} MiB of K/V per GPU per layer (BF16)")

---
## Takeaways
* Context parallelism splits the sequence; attention still needs every earlier token, so K/V blocks move between GPUs.
* With an online softmax the result is exact; memory per GPU scales with S/N, so the context can grow with the number of GPUs.
* Ring passes overlap communication with compute; all-gather is simpler for complex masks. Causal masks need load balancing.